# Compare features between PD and Control

A tutorial-style walk through the analysis in `scripts/04_compare_features.py`. Same logic, but step-by-step with explanations and intermediate output you can inspect.

**What we are testing.** Our hypothesis says that PD patients differ from healthy Controls on resting-state EEG features. We have already extracted four features per subject in `results/features.csv`:

| Column | What it measures |
|---|---|
| `alpha_peak_hz` | Frequency (Hz) of the dominant rhythm in 7–13 Hz |
| `alpha_power` | Relative power in the alpha band (8–13 Hz) |
| `theta_power` | Relative power in the theta band (4–8 Hz) |
| `beta_power` | Relative power in the beta band (13–30 Hz) |

This notebook compares these features between the two groups using a non-parametric test (Mann–Whitney U) and visualises the differences with box plots.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

# This notebook lives in `notebooks/`, so the repo root is one level up.
REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))

FEATURES_CSV = REPO_ROOT / "results" / "features.csv"
print("Features CSV exists:", FEATURES_CSV.exists())

## 2. Load the data

`features.csv` was produced by `scripts/01_extract_features.py`. It has one row per subject and includes the group label.

In [ ]:
df = pd.read_csv(FEATURES_CSV)
print(f"Total subjects: {len(df)}")
print(df.group.value_counts().to_string())
df.head()

## 3. Group descriptive statistics

For each group, look at the median and spread of each feature. This gives a first sense of where the differences might be.

In [ ]:
features = ["alpha_peak_hz", "alpha_power", "theta_power", "beta_power"]
df.groupby("group")[features].describe().T

## 4. Look at the distributions

Before running any test, plot the histograms of each feature for both groups. Visual inspection tells us the shape (skewed? bimodal?) and whether differences look real or noisy.

In [ ]:
colors = {"Control": "#0f766e", "PD": "#92400e"}

fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
for ax, feature in zip(axes, features):
    for group, color in colors.items():
        values = df.loc[df.group == group, feature]
        ax.hist(values, bins=15, alpha=0.55, color=color, label=group)
    ax.set_title(feature)
    ax.legend(fontsize=9)
    ax.set_xlabel(feature)
    ax.set_ylabel("count")
plt.tight_layout()
plt.show()

## 5. Mann–Whitney U test for one feature

Start with `alpha_peak_hz` to walk through one test.

**Why Mann–Whitney instead of a t-test?** Features derived from a power spectrum are typically skewed (not normal). Mann–Whitney makes no normality assumption: it compares the ranks of the values, so it is robust to skew and outliers.

In [ ]:
ctrl = df.loc[df.group == "Control", "alpha_peak_hz"].values
pdat = df.loc[df.group == "PD",      "alpha_peak_hz"].values

u_stat, p_value = mannwhitneyu(ctrl, pdat, alternative="two-sided")

print(f"n_Control = {len(ctrl)}, n_PD = {len(pdat)}")
print(f"Control median = {np.median(ctrl):.2f} Hz")
print(f"PD median      = {np.median(pdat):.2f} Hz")
print(f"median difference (Ctrl - PD) = {np.median(ctrl) - np.median(pdat):+.2f} Hz")
print()
print(f"U statistic = {u_stat:.1f}")
print(f"p-value     = {p_value:.4g}")

## 6. Effect size

The U statistic alone is hard to interpret because it depends on sample sizes. A more readable effect size is the **rank-biserial correlation**:

$$ r = \frac{2U}{n_1 n_2} - 1, \quad r \in [-1, 1] $$

- $r > 0$: group 1 (the first one passed to `mannwhitneyu`) had systematically larger values.
- $r < 0$: group 1 had smaller values.
- Conventional magnitudes: $|r| \approx 0.1, 0.3, 0.5$ map to small, medium, large effects.

In [ ]:
def rank_biserial(u_stat, n1, n2):
    return 2.0 * u_stat / (n1 * n2) - 1.0

r = rank_biserial(u_stat, len(ctrl), len(pdat))
print(f"rank-biserial r = {r:+.3f}")
print("Sign interpretation: positive means Control had larger values.")

## 7. Run the comparison for all four features

Wrap the test in a small function, then apply it to each feature and collect results in a tidy table.

In [ ]:
def compare_one(df, feature):
    a = df.loc[df.group == "Control", feature].values
    b = df.loc[df.group == "PD",      feature].values
    u, p = mannwhitneyu(a, b, alternative="two-sided")
    return {
        "feature":        feature,
        "median_Control": float(np.median(a)),
        "median_PD":      float(np.median(b)),
        "p_value":        float(p),
        "rank_biserial":  float(rank_biserial(u, len(a), len(b))),
    }

summary = pd.DataFrame([compare_one(df, f) for f in features])
summary

**Reading the table.** Each row is one feature. Sign of `rank_biserial`:

- positive: Control had larger values
- negative: PD had larger values

Looking at the medians and effect sizes:

- `alpha_peak_hz`: Control's median is higher ⇒ PD's alpha rhythm is **slower**. Matches hypothesis.
- `theta_power`: Control's median is lower ⇒ PD has **more theta**. Matches hypothesis.
- `beta_power`: Control's median is higher ⇒ PD has **less beta**. Matches hypothesis.
- `alpha_power`: medians are nearly equal ⇒ **no clear group difference**. This does NOT support the literal wording "alpha decreased" in our hypothesis. The decrease is in the **peak frequency**, not in the relative power of the alpha band. This is why we should reword the hypothesis as "alpha peak slowing".

## 8. Box plots for the slide deck

A single combined figure that shows all four features side by side, with the actual subject values drawn as jittered points so the reader can see the underlying distribution.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4.5))
rng = np.random.default_rng(0)

for ax, feature in zip(axes, features):
    ctrl = df.loc[df.group == "Control", feature].values
    pdat = df.loc[df.group == "PD",      feature].values

    box = ax.boxplot(
        [ctrl, pdat],
        tick_labels=["Control", "PD"],
        patch_artist=True,
        widths=0.55,
        medianprops=dict(color="black", linewidth=1.8),
        showfliers=False,
    )
    for patch, group in zip(box["boxes"], ["Control", "PD"]):
        patch.set_facecolor(colors[group])
        patch.set_alpha(0.5)
        patch.set_edgecolor(colors[group])

    for x, values, group in zip([1, 2], [ctrl, pdat], ["Control", "PD"]):
        jitter = rng.normal(0, 0.04, size=len(values))
        ax.scatter(x + jitter, values, s=14, color=colors[group],
                   alpha=0.55, edgecolors="none")

    ax.set_title(feature, fontsize=11)
    ax.grid(alpha=0.25, axis="y")
    ax.set_axisbelow(True)

fig.suptitle("PD vs Control: per-subject EEG features", y=1.02, fontsize=13)
fig.tight_layout()
plt.show()

## 9. Take-away

Three of the four features (`alpha_peak_hz`, `theta_power`, `beta_power`) show clear, significant differences in the directions predicted by the hypothesis. The relative `alpha_power` does NOT differ between groups, which is why we plan to reword the hypothesis to "alpha peak slowing" rather than "alpha decrease".

These results are independent of the Wilson–Cowan model: they establish that the data carries the hypothesised signal at the descriptive level. The model-fitting pipeline (`scripts/02` and `scripts/03`) will then re-express the same effect in terms of one biological parameter, the inhibitory time constant `tau_I`.